# 🎙️ Text to Speech (TTS) & Speech-to-Text (STT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nawooooojin/openai-api-tutorial-kr/blob/main/2_Multimodal_Generation/03-TTS_STT-Lecture.ipynb)

> **주의:** Colab 런타임을 재시작하면 `OPENAI_API_KEY`를 다시 설정해야 합니다.


## 환경 준비 및 API 키 설정

In [ ]:
%pip install -q -U "openai[realtime]"

In [1]:
# Colab에서 실행 중이라면 실습 데이터(data/)가 있는 레포를 받아옵니다.
# (로컬에서 레포를 clone 해서 실행 중이라면 그냥 넘어갑니다)
import os, sys

if "google.colab" in sys.modules and not os.path.exists("data"):
    !git clone -q --depth 1 https://github.com/nawooooojin/openai-api-tutorial-kr.git /content/openai-api-tutorial-kr
    %cd /content/openai-api-tutorial-kr/2_Multimodal_Generation

In [2]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass  # Colab이 아닌 환경
    except Exception:
        print("Colab 메뉴의 보안 비밀(열쇠 아이콘)에 OPENAI_API_KEY가 저장되어 있는지, 노트북 액세스가 켜져 있는지 확인해주세요.")

if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API 키 설정 완료")

✅ API 키 설정 완료


In [3]:
base_dirs = [
    ("data", ["image", "audio", "video"]),
    ("results", ["image", "audio", "video"]),
]

for root, subfolders in base_dirs:
    for sub in subfolders:
        path = os.path.join(root, sub)
        os.makedirs(path, exist_ok=True)
        print(f"ready: {path}")

ready: data/image
ready: data/audio
ready: data/video
ready: results/image
ready: results/audio
ready: results/video


In [4]:
from openai import OpenAI
from pathlib import Path
from IPython.display import Audio, display

client = OpenAI()
print("✅ 클라이언트 생성 완료")

✅ 클라이언트 생성 완료


## Text-to-Speech (TTS)

TTS는 텍스트를 오디오로 변환하는 모델입니다.

OpenAI API에는 13개의 내장 음성이 포함되어 있으며 다음과 같이 사용할 수 있습니다:
- 작성된 블로그 포스트를 나레이션하기
- 다국어로 음성 오디오 생성하기
- 스트리밍을 사용한 실시간 오디오 출력 제공하기

> ⚠️ Speech API의 TTS 모델(`tts-1`, `tts-1-hd`, `gpt-4o-mini-tts`)은 2027년 1월 6일 종료 예정입니다. 대체 모델(`gpt-realtime-2.1-mini`)로 음성을 만드는 방법은 TTS 파트 마지막에 있습니다.

### 주요 파라미터

**model**: 사용 가능한 TTS 모델 중 하나를 지정합니다.
- `tts-1`, `tts-1-hd`, `gpt-4o-mini-tts` 중 선택 가능합니다.

**input**: 오디오를 생성할 텍스트입니다.
- 최대 길이는 4096자입니다.

**voice**: 오디오를 생성할 때 사용할 음성입니다.
- 지원되는 음성은 `alloy`, `ash`, `ballad`, `coral`, `echo`, `fable`, `onyx`, `nova`, `sage`, `shimmer`, `verse`, `marin`, `cedar` 입니다.
- 음성의 미리듣기는 [OpenAI.fm](https://openai.fm)에서 확인할 수 있습니다.

**instructions**: 생성된 오디오의 음성을 추가 지시사항으로 제어합니다.
- 억양, 감정, 속도 등을 지정할 수 있습니다.
- `tts-1` 또는 `tts-1-hd` 모델에서는 작동하지 않습니다.

**response_format**: 오디오 출력 형식을 지정합니다.
- 지원되는 형식은 `mp3`(기본값), `opus`, `aac`, `flac`, `wav`, `pcm` 입니다.

**speed**: 생성된 오디오의 속도입니다.
- 0.25에서 4.0 사이의 값을 선택할 수 있습니다.
- 기본값은 1.0입니다.

In [5]:
from pathlib import Path
from openai import OpenAI

client = OpenAI()
speech_file_path = "results/audio/tts_sample.mp3"


with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="marin",
    input="와! 오늘 같은 날은 LLM 실습 배우기에 완전히 최고의 날이에요! 정말 신나요!",
    instructions="Speak in a cheerful and positive tone.",
) as response:
    response.stream_to_file(speech_file_path)

print("✅ 저장 완료 ->", speech_file_path)
display(Audio(speech_file_path))

✅ 저장 완료 -> results/audio/tts_sample.mp3


### Realtime API로 음성 만들기 (`gpt-realtime-2.1-mini`)
- `gpt-realtime-2.1-mini`는 Realtime API에서 사용하는 모델로, WebSocket으로 연결해 이벤트를 주고받는 방식입니다.
- 읽을 텍스트를 보내고, 들어오는 오디오 조각(`response.output_audio.delta`)을 모아 WAV 파일로 저장합니다.

In [6]:
import base64
import wave

text = "와! 오늘 같은 날은 LLM 실습 배우기에 완전히 최고의 날이에요! 정말 신나요!"
audio_chunks = bytearray()

with client.realtime.connect(model="gpt-realtime-2.1-mini") as conn:
    # 1) 세션 설정: 오디오로만 출력 + 음성 + TTS 역할 지시
    conn.session.update(session={
        "type": "realtime",
        "output_modalities": ["audio"],
        "instructions": (
            "You are a text-to-speech engine. Read the user's text aloud exactly as written, "
            "in a cheerful and positive tone. Do not add or omit any words."
        ),
        "audio": {"output": {"voice": "marin", "format": {"type": "audio/pcm", "rate": 24000}}},
    })

    # 2) 읽을 텍스트 전달 + 응답 요청
    conn.conversation.item.create(item={
        "type": "message",
        "role": "user",
        "content": [{"type": "input_text", "text": text}],
    })
    conn.response.create()

    # 3) 이벤트를 받으면서 오디오 조각 모으기
    for event in conn:
        if event.type == "response.output_audio.delta":
            audio_chunks.extend(base64.b64decode(event.delta))
        elif event.type == "response.output_audio_transcript.done":
            print("🗣️ 읽은 문장:", event.transcript)
        elif event.type == "error":
            print("❌ 오류:", event.error)
            break
        elif event.type == "response.done":
            break

# 4) PCM → WAV 저장 (mono, 16bit, 24kHz)
realtime_file_path = "results/audio/tts_realtime_sample.wav"
with wave.open(realtime_file_path, "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(24000)
    wf.writeframes(bytes(audio_chunks))

print("✅ 저장 완료 ->", realtime_file_path)
display(Audio(realtime_file_path))

🗣️ 읽은 문장: 와! 오늘 같은 날은 LLM 실습 배우기에 완전히 최고의 날이에요! 정말 신나요!


✅ 저장 완료 -> results/audio/tts_realtime_sample.wav


## Speech-to-Text (STT)

**Speech-to-Text** 는 사람의 말소리를 텍스트로 변환하는 기술입니다.

**사용 예시**
- 오디오의 언어 그대로 텍스트로 옮기는 전사 (transciption)
- 오디오를 영어로 번역 + 전사 작업
- 실시간 자막 생성
- 음성 메모 변환

**OpenAI Audio API는 두 가지 speech-to-text 엔드포인트를 제공합니다.**
- **`transcriptions`**: 오디오를 해당 언어의 텍스트로 변환
- **`translations`**: 오디오를 영어 텍스트로 번역 및 변환

**지원 모델**
- **`gpt-transcribe`**: 고품질 전사 모델
- **`gpt-live-transcribe`**: 실시간 전사 모델
- `whisper-1`, `gpt-4o-transcribe`, `gpt-4o-mini-transcribe`, `gpt-4o-transcribe-diarize`: 2027년 2월 26일 종료 예정


**파일 제한사항**
- **파일 크기**: 최대 25MB
- **지원 형식**: `mp3`, `mp4`, `mpeg`, `mpga`, `m4a`, `wav`, `webm`

**지원 언어**
- **한국어**, 영어 등 98개 언어

## 1) Transcriptions

### 기본 흐름
- `client.audio.transcriptions.create`로 오디오 파일과 모델을 지정해 텍스트를 얻습니다.
- 실습 모델: `gpt-transcribe`
- 응답은 기본적으로 JSON(`response.text`)이며, `response_format="text"`로 순수 텍스트를 받을 수 있습니다.

### 주요 파라미터
- **`file`**: 변환할 오디오 파일 객체 (파일 이름이 아님)
- **`model`**: 사용할 모델 (`gpt-transcribe` 등)
- **`language`**: 입력 오디오의 언어 (ISO-639-1 형식은 정밀도와 지연을 개선합니다, 예: `"ko"`, `"en"`)
    - [ISO-639-1 형식](https://en.wikipedia.org/wiki/List_of_ISO_639_language_codes)
- **`prompt`**: (선택) 모델의 스타일을 안내하거나 이전 오디오 세그먼트를 계속하기 위한 텍스트 등
- **`keywords`**: (선택) 잘 틀리는 고유명사·전문용어 리스트
- **`response_format`**: 출력 형식 (`json`, `text`, `srt`, `verbose_json`, `vtt`, `diarized_json`)
- **`temperature`**: 0~1 사이의 샘플링 temperature (기본값: 0)


In [7]:
from openai import OpenAI

client = OpenAI()
audio_file= open("data/audio/gasimbi.mp3", "rb")

transcription = client.audio.transcriptions.create(
    model="gpt-transcribe",
    file=audio_file
)

print(transcription.text)

난 몇 달 전만 해도 가심비가 가성비보다 중요하다고 생각해서 뭐 살 때 그리 고민하지 않았는데 요즘은 달라.


**기본 응답 타입 (json) 예시**

In [8]:
print(transcription.model_dump_json(indent=2))

{
  "text": "난 몇 달 전만 해도 가심비가 가성비보다 중요하다고 생각해서 뭐 살 때 그리 고민하지 않았는데 요즘은 달라.",
  "languages": [
    {
      "code": "ko"
    }
  ],
  "logprobs": null,
  "usage": {
    "seconds": 10.0,
    "type": "duration"
  }
}


## 2) Translation
- `client.audio.translations.create`로 오디오 파일을 입력받아 영어 텍스트로 번역 및 전사합니다.
- 실습 모델: `whisper-1`
- `/Transcriptions` 엔드포인트와 달리, 출력은 원본 언어가 아닌 **영어 텍스트**로 변환됩니다.

**주요 특징**
- **입력**: 지원되는 모든 언어의 오디오 파일
- **출력**: 영어로 번역된 텍스트
- **모델**: `whisper-1` 모델만 지원
- ⚠️ `whisper-1`과 translations 엔드포인트는 2027년 2월 26일 종료 예정입니다.

In [9]:
from openai import OpenAI

client = OpenAI()
audio_file = open("data/audio/convenience_store.mp3", "rb")

translation = client.audio.translations.create(
    model="whisper-1",
    file=audio_file,
)

print(translation.text)

That's right. Sometimes, there are places where a lot of good customers come. In the case of convenience stores, there are people who give one plus one product each.


## 3) 고급 옵션: 응답 포맷 · 프롬프트

### 응답 포맷 (response_format)
- **`json`**: 기본 JSON 형식 (텍스트 포함)
- **`text`**: 순수 텍스트만 반환
- **`verbose_json`**: 타임스탬프 등 추가 메타데이터 포함
- **`diarized_json`**: 발화자 구분 정보 포함 (gpt-4o-transcribe-diarize 모델 전용)
- **`srt`**: 자막 파일 형식
- **`vtt`**: WebVTT 자막 형식
- `gpt-transcribe`는 `json`, `text`만 지원합니다.

### 프롬프트 (prompt)
- 모델이 잘못 인식하는 용어/고유명사를 안내해 정확도 향상
- 이전 오디오 세그먼트의 텍스트를 전달해 맥락 유지


### 텍스트 포맷 실습

In [10]:
from openai import OpenAI

client = OpenAI()
audio_file = open("data/audio/doseguon.mp3", "rb")

transcription = client.audio.transcriptions.create(
    model="gpt-transcribe",
    file=audio_file,
    response_format="text"
)

print(transcription)


내가 오늘은 도색권이라는 걸 알게 됐는데, 도서반하고 역세권의 합성어로 도서반이 가까운 거리에 있는 곳을 뜻해.



### Prompting 실습

`prompt` 파라미터를 활용하여 전사 품질을 향상시킬 수 있습니다.

**주요 활용 사례**

- **고유명사/약어 오인식 개선**:
    - 예: DALL·E, GPT-3가 "GPT 3", "DALI"로 전사될 경우
    - 프롬프트: `"The transcript is about OpenAI which makes model like DALL·E, GPT-3, and ChatGPT..."`

- **세그먼트 간 문맥 유지**:

- **문장부호 포함 유도**:
    - 예: `"Hello, welcome to my lecture."`

- **말채움(filler) 단어 유지**:
    - 예: `"Umm, let me think like, hmm... Okay, here's what I'm, like, thinking."`

- **표기 체계 지정**:
    - 간체/번체 중국어 등


In [11]:
from openai import OpenAI

client = OpenAI()
audio_file = open("data/audio/lol.mp3", "rb")

transcription = client.audio.transcriptions.create(
  model="gpt-transcribe",
  file=audio_file,
  response_format="text",
)

print(transcription)


롤 할래? 이번 패치에서 너 주챔에시 버프됐더라.



In [12]:
from openai import OpenAI

client = OpenAI()
audio_file = open("data/audio/lol.mp3", "rb")

transcription = client.audio.transcriptions.create(
  model="gpt-transcribe",
  file=audio_file,
  response_format="text",
  prompt = "이 대화는 리그 오브 레전드(롤) 게임에 대한 대화입니다. 주챔은 주요 챔피언의 줄임말이고, 챔피언 이름은 애쉬, 그라가스, 티모 등이 있습니다."
)

print(transcription)


롤 할래? 이번 패치에서 너 주챔 애쉬 버프됐더라.



In [13]:
# keywords로 잘 틀리는 단어 직접 알려주기
with open("data/audio/doseguon.mp3", "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=audio_file,
        response_format="text",
        keywords=["도세권", "역세권", "도서관"],
    )

print(transcription)

내가 오늘은 도세권이라는 걸 알게 됐는데, 도서관하고 역세권의 합성어로 도서관이 가까운 거리에 있는 곳을 뜻해.



### Post-processing with GPT

전사 품질 향상을 위해 GPT 모델을 활용한 후처리(post-processing)를 수행할 수 있습니다.

- **방법**: Responses API를 사용해 전사된 텍스트를 GPT 모델에 입력하고, 오타 수정 및 문맥 개선을 요청합니다.
- **system_prompt 활용**: `system_prompt`에 회사명, 제품명 등 고유명사를 명시하여 정확도를 높일 수 있습니다.
- **장점**: `prompt` 파라미터와 유사하지만, 더 복잡한 지시사항과 맥락을 제공할 수 있습니다.

In [14]:
system_prompt = """
당신은 게임 대화를 전사하는 도우미입니다. 전사된 텍스트의 오타를 수정하고,
리그 오브 레전드(롤) 게임의 용어를 올바르게 교정해주세요.
다음 용어들이 올바르게 표기되었는지 확인하세요: 롤, 리그 오브 레전드, 주챔(주요 챔피언),
애쉬, 그라가스, 티모, 버프 등. 필요한 구두점(마침표, 쉼표)과 대문자를 추가하되,
제공된 맥락만 사용하세요.
"""


def generate_corrected_transcript(audio_file_path, reasoning_effort="medium", verbosity="medium", system_prompt=system_prompt):
    # 1단계: 오디오를 텍스트로 전사
    audio_file = open(audio_file_path, "rb")
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=audio_file,
        response_format="text"
    )
    audio_file.close()

    # 2단계: 전사된 텍스트를 GPT로 후처리
    response = client.responses.create(
        model="gpt-6-luna",
        reasoning={"effort": reasoning_effort},
        text={"verbosity": verbosity},
        input=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": f"다음 전사 텍스트를 교정해주세요:\n{transcription}"
            }
        ]
    )

    return response.output_text, transcription

# 롤 게임 오디오 파일로 STT 및 후처리
audio_file_path = "data/audio/lol.mp3"
corrected_text, original_text = generate_corrected_transcript(audio_file_path)

print("원본 전사:")
print(original_text)
print("\n교정된 텍스트:")
print(corrected_text)

원본 전사:
롤 할래? 이번 패치에서 너 주챔 애쉬 버프됐더라.


교정된 텍스트:
롤 할래? 이번 패치에서 네 주챔 애쉬 버프됐더라.


### 참고 자료
- https://platform.openai.com/docs/api-reference/audio/createSpeech
- https://platform.openai.com/docs/guides/text-to-speech
- https://openai.fm
- https://platform.openai.com/docs/guides/speech-to-text
- https://platform.openai.com/docs/api-reference/audio
- 실습 오디오 데이터
    - **데이터셋**: AI Hub 연령대별 특징적 발화(은어·속어 등) 음성 데이터
    - **AI Hub**: [AI Hub](https://aihub.or.kr)
